In [1]:
import re, gc, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda")
MAX_LEN, BATCH, ACCUM, EPOCHS, LR = 512, 8, 2, 3, 2e-5
tok = AutoTokenizer.from_pretrained("microsoft/codebert-base")
RESULTS = "../results/codebert_cross_project.csv"
os.makedirs("../results", exist_ok=True)

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

df = pd.read_json("../data/function.json")
df["clean"] = df["func"].apply(clean_code)
qemu = df[df["project"] == "qemu"]
ffmpeg = df[df["project"] == "FFmpeg"]

def make_loader(d, shuffle):
    items = list(zip(d["clean"], d["target"]))
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(labels)
        return enc
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)

def run(train_df, test_df, name):
    torch.manual_seed(42); np.random.seed(42)
    train_dl, test_dl = make_loader(train_df, True), make_loader(test_df, False)
    model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).to(device)
    counts = np.bincount(train_df["target"].values)
    weights = torch.tensor(len(train_df) / (2 * counts), dtype=torch.float32, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(train_dl) // ACCUM
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for i, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
                loss = loss_fn(logits.float(), b["labels"]) / ACCUM
            scaler.scale(loss).backward()
            if (i + 1) % ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            if i % 500 == 0:
                print(f"[{name}] epoch {epoch} step {i}/{len(train_dl)} loss {loss.item() * ACCUM:.3f}")

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in test_dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(b["labels"].cpu().numpy())
    y, p = np.concatenate(labels), np.concatenate(preds)
    res = dict(name=name, n_train=len(train_df), n_test=len(test_df),
               acc=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
               recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0))
    pd.DataFrame([res]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print(res)

    del model, opt, sched, scaler, loss_fn
    gc.collect(); torch.cuda.empty_cache()

In [2]:
f_tr, f_te = train_test_split(ffmpeg, test_size=0.2, random_state=42, stratify=ffmpeg["target"])
q_tr, q_te = train_test_split(qemu, test_size=0.2, random_state=42, stratify=qemu["target"])

scenarios = [
    ("QEMU -> FFmpeg", qemu, ffmpeg),
    ("FFmpeg -> QEMU", ffmpeg, qemu),
    ("QEMU -> QEMU", q_tr, q_te),
    ("FFmpeg -> FFmpeg", f_tr, f_te),
]

done = set(pd.read_csv(RESULTS)["name"]) if os.path.exists(RESULTS) else set()
for name, tr, te in scenarios:
    if name not in done:
        run(tr, te, name)

print(pd.read_csv(RESULTS))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[QEMU -> FFmpeg] epoch 1 step 0/2194 loss 0.672
[QEMU -> FFmpeg] epoch 1 step 500/2194 loss 0.672
[QEMU -> FFmpeg] epoch 1 step 1000/2194 loss 0.629
[QEMU -> FFmpeg] epoch 1 step 1500/2194 loss 0.621
[QEMU -> FFmpeg] epoch 1 step 2000/2194 loss 0.797
[QEMU -> FFmpeg] epoch 2 step 0/2194 loss 0.595
[QEMU -> FFmpeg] epoch 2 step 500/2194 loss 0.755
[QEMU -> FFmpeg] epoch 2 step 1000/2194 loss 0.646
[QEMU -> FFmpeg] epoch 2 step 1500/2194 loss 0.470
[QEMU -> FFmpeg] epoch 2 step 2000/2194 loss 0.594
[QEMU -> FFmpeg] epoch 3 step 0/2194 loss 0.413
[QEMU -> FFmpeg] epoch 3 step 500/2194 loss 0.710
[QEMU -> FFmpeg] epoch 3 step 1000/2194 loss 0.612
[QEMU -> FFmpeg] epoch 3 step 1500/2194 loss 0.549
[QEMU -> FFmpeg] epoch 3 step 2000/2194 loss 0.506
{'name': 'QEMU -> FFmpeg', 'n_train': 17549, 'n_test': 9769, 'acc': 0.5282014535776436, 'precision': 0.558601134215501, 'recall': 0.35595261995583216, 'f1': 0.4348252605763335}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[FFmpeg -> QEMU] epoch 1 step 0/1222 loss 0.637
[FFmpeg -> QEMU] epoch 1 step 500/1222 loss 0.626
[FFmpeg -> QEMU] epoch 1 step 1000/1222 loss 0.662
[FFmpeg -> QEMU] epoch 2 step 0/1222 loss 0.672
[FFmpeg -> QEMU] epoch 2 step 500/1222 loss 0.664
[FFmpeg -> QEMU] epoch 2 step 1000/1222 loss 0.732
[FFmpeg -> QEMU] epoch 3 step 0/1222 loss 0.618
[FFmpeg -> QEMU] epoch 3 step 500/1222 loss 0.552
[FFmpeg -> QEMU] epoch 3 step 1000/1222 loss 0.558
{'name': 'FFmpeg -> QEMU', 'n_train': 9769, 'n_test': 17549, 'acc': 0.5571827454555816, 'precision': 0.4721161191749427, 'recall': 0.3305254713196951, 'f1': 0.3888320880849391}


... (2 outputs left out)

[QEMU -> QEMU] epoch 1 step 0/1755 loss 0.742
[QEMU -> QEMU] epoch 1 step 500/1755 loss 0.793
[QEMU -> QEMU] epoch 1 step 1000/1755 loss 0.803
[QEMU -> QEMU] epoch 1 step 1500/1755 loss 0.745
[QEMU -> QEMU] epoch 2 step 0/1755 loss 0.603
[QEMU -> QEMU] epoch 2 step 500/1755 loss 0.644
[QEMU -> QEMU] epoch 2 step 1000/1755 loss 0.433
[QEMU -> QEMU] epoch 2 step 1500/1755 loss 0.859
[QEMU -> QEMU] epoch 3 step 0/1755 loss 0.712
[QEMU -> QEMU] epoch 3 step 500/1755 loss 0.736
[QEMU -> QEMU] epoch 3 step 1000/1755 loss 0.491
[QEMU -> QEMU] epoch 3 step 1500/1755 loss 0.547
{'name': 'QEMU -> QEMU', 'n_train': 14039, 'n_test': 3510, 'acc': 0.6660968660968661, 'precision': 0.632569558101473, 'recall': 0.5167112299465241, 'f1': 0.5688005886681383}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[FFmpeg -> FFmpeg] epoch 1 step 0/977 loss 0.570
[FFmpeg -> FFmpeg] epoch 1 step 500/977 loss 0.650
[FFmpeg -> FFmpeg] epoch 2 step 0/977 loss 0.664
[FFmpeg -> FFmpeg] epoch 2 step 500/977 loss 0.734
[FFmpeg -> FFmpeg] epoch 3 step 0/977 loss 0.638
[FFmpeg -> FFmpeg] epoch 3 step 500/977 loss 0.727
{'name': 'FFmpeg -> FFmpeg', 'n_train': 7815, 'n_test': 1954, 'acc': 0.6151484135107472, 'precision': 0.6334792122538293, 'recall': 0.5813253012048193, 'f1': 0.606282722513089}
               name  n_train  n_test       acc  precision    recall        f1
0    QEMU -> FFmpeg    17549    9769  0.528201   0.558601  0.355953  0.434825
1    FFmpeg -> QEMU     9769   17549  0.557183   0.472116  0.330525  0.388832
2      QEMU -> QEMU    14039    3510  0.666097   0.632570  0.516711  0.568801
3  FFmpeg -> FFmpeg     7815    1954  0.615148   0.633479  0.581325  0.606283
